In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
import datetime
from keras.datasets import fashion_mnist
import wandb

In [2]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNetwork

In [3]:
def normalize(x):
    return x.reshape(len(x), -1).astype('float64') / (np.max(x) - np.min(x))

In [4]:
def load_and_prepare_data(dataset="fashion_mnist"):
    # Load the Fashion MNIST dataset
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    
    # Using train_test_split to separate validation data (10% of training data)
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=69)
    
    # Normalize image pixel values
    x_train = normalize(x_train)
    x_val   = normalize(x_val)
    x_test  = normalize(x_test)
    
    # Determine the number of classes from the unique labels
    classes = np.unique(y_train)
    num_classes = len(classes)
    
    # One-hot encode labels based on the discovered number of classes
    y_train = np.eye(num_classes)[y_train]
    y_val   = np.eye(num_classes)[y_val]
    y_test  = np.eye(num_classes)[y_test]
    
    return x_train, y_train, x_val, y_val, x_test, y_test

In [5]:
def train_and_evaluate(config=None):
    with wandb.init(config=config):
        cfg = wandb.config
        
        # Load and prepare the dataset
        x_train, y_train, x_val, y_val, x_test, y_test = load_and_prepare_data()
        
        # Model with configuration parameters
        model = NeuralNetwork(
            input_size = x_train.shape[1],
            num_classes = y_train.shape[1],
            num_hidden = cfg.num_layers,
            hidden_units = cfg.hidden_size,
            init_method = cfg.weight_init,
            activation = cfg.activation,
            loss_fn = cfg.loss,
            epochs = cfg.epochs,
            batch_size = cfg.batch_size,
            optimizer = cfg.optimizer,
            lr = cfg.learning_rate,
            weight_decay = cfg.weight_decay,
            momentum = cfg.momentum if hasattr(cfg, 'momentum') else 0.9,
            beta = cfg.beta if hasattr(cfg, 'beta') else 0.9,
            beta1 = cfg.beta1 if hasattr(cfg, 'beta1') else 0.9,
            beta2 = cfg.beta2 if hasattr(cfg, 'beta2') else 0.999,
            epsilon = cfg.epsilon if hasattr(cfg, 'epsilon') else 1e-6
        )
        
        # Train the model using the training and validation data
        model.fit(x_train, y_train, x_val, y_val)
        
        # Evaluate on validation set
        val_preds = model.predict(x_val.T)
        val_loss  = model.compute_loss(val_preds, y_val)
        val_acc   = model.accuracy(val_preds, y_val)
        
        # Evaluate on test set
        test_preds = model.predict(x_test.T)
        test_loss  = model.compute_loss(test_preds, y_test)
        test_acc   = model.accuracy(test_preds, y_test)
        
        # Log evaluation metrics to wandb with a timestamp
        wandb.log({
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "created": datetime.datetime.now().isoformat()
        })

In [6]:
sweep_config = {
    'method': 'bayes',
    'name': 'Bayesian_sweep_cross_entropy',
    'metric': {'name': 'validation_accuracy', 'goal': 'maximize'},
    'parameters': {
        'epochs': {'values': [5, 10]},
        'num_layers': {'values': [3, 4, 5]},
        'hidden_size': {'values': [32, 64, 128]},
        'weight_decay': {'values': [0, 0.0005, 0.5]},
        'learning_rate': {'values': [0.001, 0.0001]},
        'optimizer': {'values': ['sgd']},
        'batch_size': {'values': [16, 32, 64]},
        'weight_init': {'values': ['Random', 'Xavier']},
        'activation': {'values': ['Sigmoid', 'Tanh', 'ReLU']},
        'loss': {'values': ['cross_entropy']}
    }
}

In [7]:
def run_experiment():
    sweep_id = wandb.sweep(sweep_config, project="fashion-mnist-classification")
    wandb.agent(sweep_id, function=train_and_evaluate, count=10)
    wandb.finish()

In [8]:
if __name__ == "__main__":
    run_experiment()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Create sweep with ID: 2tvmzjn4
Sweep URL: https://wandb.ai/mrsagarbiswas-iit-madras/fashion-mnist-classification/sweeps/2tvmzjn4


wandb: Agent Starting Run: cyrjxqnu with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_size: 64
wandb: 	learning_rate: 0.0001
wandb: 	loss: cross_entropy
wandb: 	num_layers: 4
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0.5
wandb: 	weight_init: Random
wandb: Currently logged in as: mrsagarbiswas (mrsagarbiswas-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1: train_loss = 2.10, valid_loss = 2.17, train_accuracy = 0.44, val_accuracy = 0.43
Epoch 2: train_loss = 1.34, valid_loss = 1.39, train_accuracy = 0.55, val_accuracy = 0.54
Epoch 3: train_loss = 1.06, valid_loss = 1.08, train_accuracy = 0.63, val_accuracy = 0.62
Epoch 4: train_loss = 0.92, valid_loss = 0.93, train_accuracy = 0.66, val_accuracy = 0.65
Epoch 5: train_loss = 0.83, valid_loss = 0.83, train_accuracy = 0.68, val_accuracy = 0.68
Epoch 6: train_loss = 0.78, valid_loss = 0.78, train_accuracy = 0.71, val_accuracy = 0.70
Epoch 7: train_loss = 0.75, valid_loss = 0.75, train_accuracy = 0.72, val_accuracy = 0.73
Epoch 8: train_loss = 0.73, valid_loss = 0.73, train_accuracy = 0.73, val_accuracy = 0.74
Epoch 9: train_loss = 0.72, valid_loss = 0.71, train_accuracy = 0.74, val_accuracy = 0.74
Epoch 10: train_loss = 0.70, valid_loss = 0.70, train_accuracy = 0.75, val_accuracy = 0.76


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▃▅▆▆▇▇███
train_loss,█▄▃▂▂▁▁▁▁▁
val_accuracy,▁▃▅▆▆▇▇████
val_loss,█▄▃▂▂▁▁▁▁▁▁
created,2025-03-10T16:40:26....
epoch,9
test_accuracy,0.7446
test_loss,0.71993


...

Epoch 1: train_loss = 0.70, valid_loss = 0.70, train_accuracy = 0.76, val_accuracy = 0.76
Epoch 2: train_loss = 0.66, valid_loss = 0.67, train_accuracy = 0.78, val_accuracy = 0.78
Epoch 3: train_loss = 0.65, valid_loss = 0.66, train_accuracy = 0.78, val_accuracy = 0.78
Epoch 4: train_loss = 0.65, valid_loss = 0.65, train_accuracy = 0.79, val_accuracy = 0.78
Epoch 5: train_loss = 0.64, valid_loss = 0.64, train_accuracy = 0.79, val_accuracy = 0.79
Epoch 6: train_loss = 0.64, valid_loss = 0.64, train_accuracy = 0.79, val_accuracy = 0.79
Epoch 7: train_loss = 0.64, valid_loss = 0.64, train_accuracy = 0.79, val_accuracy = 0.79
Epoch 8: train_loss = 0.63, valid_loss = 0.64, train_accuracy = 0.79, val_accuracy = 0.79
Epoch 9: train_loss = 0.63, valid_loss = 0.64, train_accuracy = 0.79, val_accuracy = 0.79
Epoch 10: train_loss = 0.63, valid_loss = 0.63, train_accuracy = 0.79, val_accuracy = 0.79


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▅▇▇▇█████
train_loss,█▄▃▃▂▂▂▁▁▁
val_accuracy,▁▅▇▇███████
val_loss,█▄▃▃▂▂▁▁▁▁▁
created,2025-03-10T16:43:40....
epoch,9
test_accuracy,0.7821
test_loss,0.6489


wandb: Agent Starting Run: ortwy7he with config:
wandb: 	activation: ReLU
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_size: 32
wandb: 	learning_rate: 0.0001
wandb: 	loss: cross_entropy
wandb: 	num_layers: 5
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0.0005
wandb: 	weight_init: Xavier


Epoch 1: train_loss = 1.03, valid_loss = 1.02, train_accuracy = 0.66, val_accuracy = 0.67
Epoch 2: train_loss = 0.67, valid_loss = 0.67, train_accuracy = 0.76, val_accuracy = 0.76
Epoch 3: train_loss = 0.59, valid_loss = 0.59, train_accuracy = 0.80, val_accuracy = 0.80
Epoch 4: train_loss = 0.54, valid_loss = 0.55, train_accuracy = 0.81, val_accuracy = 0.82
Epoch 5: train_loss = 0.51, valid_loss = 0.51, train_accuracy = 0.83, val_accuracy = 0.83
Epoch 6: train_loss = 0.48, valid_loss = 0.49, train_accuracy = 0.83, val_accuracy = 0.83
Epoch 7: train_loss = 0.46, valid_loss = 0.48, train_accuracy = 0.84, val_accuracy = 0.84
Epoch 8: train_loss = 0.45, valid_loss = 0.46, train_accuracy = 0.84, val_accuracy = 0.84
Epoch 9: train_loss = 0.43, valid_loss = 0.45, train_accuracy = 0.85, val_accuracy = 0.85
Epoch 10: train_loss = 0.42, valid_loss = 0.44, train_accuracy = 0.85, val_accuracy = 0.85


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▅▆▇▇▇▇███
train_loss,█▄▃▂▂▂▁▁▁▁
val_accuracy,▁▅▆▇▇▇█████
val_loss,█▄▃▂▂▂▁▁▁▁▁
created,2025-03-10T16:44:10....
epoch,9
test_accuracy,0.8378
test_loss,0.46198
